# 三个 Voter 训练与硬投票合并

Spaceship Titanic 项目的核心代码 —— 三个 CatBoost 模型用不同的特征工程方案分别训练，最后通过硬投票（majority vote）合并为最终预测，Public LB 0.81529。

| Voter | 中文名 | FE 风格 | 调参方式 | 训练协议 | 单飞 LB |
| --- | --- | --- | --- | --- | --- |
| Voter 1 | Lean（极简派）| 最少特征，丢掉 Name | Optuna TPE 40 trials | 5 seed × 5 fold = 25 拟合 | 0.81365 |
| Voter 2 | Rich（最大化派）| 保留 Name + 随机抽样填补 + 选 top-27 | RandomSearch | 85/15 单切分 | 0.81458 |
| Voter 3 | Reference（参考基线）| 基础 FE，最简单 | 手写默认值 | 70/30 单切分 | 0.81248 |
| 合并 | Majority Vote | — | — | 2-of-3 多数票 | **0.81529** |

**运行方式**：把 `train.csv` 和 `test.csv` 放到本 notebook 同目录，自上而下 Run All 即可。

## 0. 公共配置

导入依赖、定义共享常量。`RUN_VOTERS_FROM_SCRATCH = True` 强制重训所有 voter；设为 `False` 时如果 submission CSV 已存在就跳过。

In [ ]:
# 标准库
import os
import time
import numpy as np
import pandas as pd

# sklearn 工具
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# CatBoost 是三个 voter 唯一用的模型
from catboost import CatBoostClassifier

# 五个原始消费列（贯穿三个 voter 的特征工程）
SPEND_COLS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# 控制是否强制重训
RUN_VOTERS_FROM_SCRATCH = True


## 1. Voter 1 — Lean CatBoost（极简 FE + Optuna 调参）

设计理念：**只保留最强信号**。
- 砍掉：Name、Surname、AgeBin、CabinNumBin
- 保留：Cabin 拆分、Group 大小、Spend 聚合、Is_Child、log-spend
- 调参：Optuna TPE（已离线跑完 40 trials，最优参数写死在下面）
- 训练协议：**5 个 random seed × 5 折 = 25 次拟合**，结果概率平均
- 思路：高学习率 + 弱正则，让模型敢学，再靠多次拟合平均稳住

In [ ]:
# Optuna 离线搜出来的最优超参（lr 偏激进 + l2 偏弱）
V9_BEST_PARAMS = dict(
    learning_rate=0.07485688984060579,
    depth=6,
    l2_leaf_reg=1.316777848260815,
    bagging_temperature=0.11773270507332459,
    random_strength=8.488957933498611,
    border_count=205,
)

# 5 个 seed —— 平均掉单个 seed 带来的随机偏差
V9_SEEDS = [42, 7, 2024, 1234, 99]


def v9_split_cabin(value):
    """Cabin 字段拆成 Deck / Num / Side 三列"""
    if pd.isna(value):
        return 'U', np.nan, 'U'
    parts = str(value).split('/')
    if len(parts) != 3:
        return 'U', np.nan, 'U'
    deck, num, side = parts
    try:
        num = int(num)
    except ValueError:
        num = np.nan
    return deck, num, side


def v9_feature_engineer(train_df, test_df):
    """Voter 1 的极简 FE：只保留最强信号，全部 one-hot + 数值化"""
    out = []
    for df in [train_df.copy(), test_df.copy()]:
        # 拆 Cabin
        cab = df['Cabin'].apply(v9_split_cabin)
        df['Deck'] = cab.apply(lambda x: x[0])
        df['Num']  = cab.apply(lambda x: x[1])
        df['Side'] = cab.apply(lambda x: x[2])
        # 从 PassengerId 前缀提取 Group 编号
        df['Group'] = df['PassengerId'].astype(str).str.split('_').str[0]
        out.append(df)
    df_tr_v, df_te_v = out

    # HomePlanet / Destination：用同组众数填，再用全局众数兜底
    combined = pd.concat([df_tr_v, df_te_v], ignore_index=True)
    for col in ['HomePlanet', 'Destination']:
        grp_mode = combined.groupby('Group')[col].transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
        combined[col] = combined[col].fillna(grp_mode)
        combined[col] = combined[col].fillna(combined[col].mode()[0])
    n_train = len(df_tr_v)
    df_tr_v[['HomePlanet','Destination']] = combined.iloc[:n_train][['HomePlanet','Destination']].values
    df_te_v[['HomePlanet','Destination']]  = combined.iloc[n_train:][['HomePlanet','Destination']].values

    # Spend 相关：缺失填 0，再加聚合特征
    for df in [df_tr_v, df_te_v]:
        df[SPEND_COLS] = df[SPEND_COLS].fillna(0)
        df['TotalSpend']  = df[SPEND_COLS].sum(axis=1)
        df['NoSpending']  = (df['TotalSpend'] == 0).astype(int)  # 完全不消费的标志
        df['LuxurySpend'] = df['Spa'] + df['VRDeck']             # 奢侈消费

    # CryoSleep：有消费的人一定没冷冻
    for df in [df_tr_v, df_te_v]:
        df.loc[df['CryoSleep'].isna() & (df['TotalSpend'] > 0), 'CryoSleep'] = False
    cryo_mode = df_tr_v['CryoSleep'].mode()[0]

    # 其他缺失值：median / mode 兜底，再加几个派生列
    for df in [df_tr_v, df_te_v]:
        df['CryoSleep'] = df['CryoSleep'].fillna(cryo_mode)
        df['Age'] = df['Age'].fillna(df_tr_v['Age'].median())
        df['VIP'] = df['VIP'].fillna(df_tr_v['VIP'].mode()[0])
        num = pd.to_numeric(df['Num'], errors='coerce')
        df['Num'] = num.fillna(num.median())
        df['Group_Size'] = df.groupby('Group')['Group'].transform('count')
        df['Is_Child'] = (df['Age'] <= 12).astype(int)
        # 消费列加 log 版本（消费分布右偏严重）
        for c in SPEND_COLS + ['TotalSpend']:
            df[f'{c}_Log'] = np.log1p(df[c])

    # 丢掉非建模列
    drop_cols = ['PassengerId', 'Cabin', 'Name', 'Group']
    test_id_v = df_te_v['PassengerId'].copy()
    df_tr_v = df_tr_v.drop(columns=drop_cols)
    df_te_v = df_te_v.drop(columns=drop_cols)

    # 类别列 one-hot
    df_tr_v = pd.get_dummies(df_tr_v, columns=['HomePlanet','Destination','Side','Deck'])
    df_te_v = pd.get_dummies(df_te_v,  columns=['HomePlanet','Destination','Side','Deck'])
    # 训练集见过的列，测试集要对齐
    df_te_v = df_te_v.reindex(
        columns=[c for c in df_tr_v.columns if c != 'Transported'],
        fill_value=False)
    # bool 转 int
    for col in ['CryoSleep', 'VIP']:
        df_tr_v[col] = df_tr_v[col].astype(int)
        df_te_v[col] = df_te_v[col].astype(int)
    return df_tr_v, df_te_v, test_id_v


In [ ]:
# 训练 Voter 1：5 seed × 5 fold = 25 次拟合
if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('submission_v9.csv'):
    # 读原始数据
    train_raw_v9 = pd.read_csv('train.csv')
    test_raw_v9  = pd.read_csv('test.csv')
    df_tr_v9, df_te_v9, test_id_v9 = v9_feature_engineer(train_raw_v9, test_raw_v9)

    y_v9 = df_tr_v9['Transported'].astype(int).reset_index(drop=True)
    X_v9 = df_tr_v9.drop(columns=['Transported']).reset_index(drop=True)
    X_v9_test = df_te_v9.reset_index(drop=True)

    # 标准化（CatBoost 其实不强制要，但保留与原 v9.py 一致）
    sc_v9 = StandardScaler()
    X_v9s      = pd.DataFrame(sc_v9.fit_transform(X_v9), columns=X_v9.columns)
    X_v9s_test = pd.DataFrame(sc_v9.transform(X_v9_test), columns=X_v9.columns)

    # 合并 Optuna 最优超参 + 训练相关固定值
    cb_p = dict(V9_BEST_PARAMS, iterations=3000,
                loss_function='Logloss', eval_metric='Accuracy', verbose=0)

    # 累加每次拟合的概率，最后除以总次数
    t0 = time.time()
    test_avg_v9 = np.zeros(len(X_v9_test))
    n_fits = len(V9_SEEDS) * 5
    for si, seed in enumerate(V9_SEEDS):
        skf_v9 = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        for fold, (tr_i, va_i) in enumerate(skf_v9.split(X_v9s, y_v9), start=1):
            cb_v9 = CatBoostClassifier(**dict(cb_p, random_seed=seed),
                                       allow_writing_files=False)
            cb_v9.fit(X_v9s.iloc[tr_i], y_v9.iloc[tr_i],
                      eval_set=(X_v9s.iloc[va_i], y_v9.iloc[va_i]),
                      early_stopping_rounds=100, verbose=0)
            test_avg_v9 += cb_v9.predict_proba(X_v9s_test)[:, 1] / n_fits
        print(f'  seed {seed} 完成   （已耗时 {(time.time()-t0)/60:.1f} 分钟）')

    # 平均概率阈值 0.5 转 bool 标签，存盘
    v9_pred = (test_avg_v9 > 0.5).astype(bool)
    pd.DataFrame({'PassengerId': test_id_v9, 'Transported': v9_pred}
                 ).to_csv('submission_v9.csv', index=False)
    print(f'\nVoter 1 完成，已保存 submission_v9.csv （{test_avg_v9.shape[0]} 行）')
else:
    print('submission_v9.csv 已存在，跳过重训。')
    print('如需强制重训，请把第 0 节的 RUN_VOTERS_FROM_SCRATCH 改为 True。')


## 2. Voter 2 — Rich CatBoost（丰富 FE + Name 编码 + Top-27 选择）

设计理念：**信号越多越好，但模型必须压住**。
- 保留 Name 字段，把 first / last 各 LabelEncode 一次（捕捉同姓家庭信号）
- 缺失值用列内**随机抽样**填，注入训练时的随机性
- 跑一次 baseline CatBoost 测特征重要性，**只留 top-27**
- 调参：RandomSearch（已离线跑完，最优参数写死）
- 训练协议：85/15 单切分，单次拟合
- 哲学：低学习率 + 强正则，正好和 Voter 1 相反

In [ ]:
# RandomSearch 离线搜出来的最优超参（lr 偏保守 + l2 偏强，正好和 V9 相反）
P5T6_PARAMS = dict(
    learning_rate=0.018049356549743555,
    depth=6,
    l2_leaf_reg=7.50,
    border_count=182,
    random_seed=42,
)


def p5t6_prepare(seed=42):
    """Voter 2 的丰富 FE + 随机抽样填补"""
    rng = np.random.RandomState(seed)

    df_tr = pd.read_csv('train.csv')
    df_te = pd.read_csv('test.csv')
    df = pd.concat([df_tr, df_te], axis=0)

    base_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpending'] = df[base_cols].sum(axis=1)

    # 拆 Cabin
    cab = df['Cabin'].apply(
        lambda x: x.split('/') if not isinstance(x, float) else ['-1','-1','-1']).tolist()
    cab = np.array(cab)
    df['Cabin_deck'] = cab[:, 0]
    df['Cabin_num']  = cab[:, 1]
    df['Cabin_side'] = cab[:, 2]
    df.drop(columns='Cabin', inplace=True)

    # Group size
    grp = df['PassengerId'].apply(lambda x: x.split('_')[0]).value_counts().to_dict()
    df['Group_size'] = df['PassengerId'].apply(lambda x: grp[x.split('_')[0]])
    df.set_index('PassengerId', inplace=True)

    # 拆 Name 为 first / last
    df['Name'] = df['Name'].fillna('Unk Unk')
    parts = np.array(df['Name'].apply(lambda x: x.split(' ')).tolist())
    df['Name_first'] = parts[:, 0]
    df['Name_last']  = parts[:, 1]
    df.drop(columns=['Name'], inplace=True)

    # HomePlanet / Destination：按列内值频率随机抽样填
    for c in ['HomePlanet', 'Destination']:
        vc = df[c].value_counts()
        df.loc[df[c].isna(), c] = rng.choice(
            vc.index, df[c].isna().sum(), p=(vc / vc.sum()).values)

    # Cabin：deck 在 F/G 中随机选，num 用均值，side 在 S/P 中随机选
    n_miss = (df['Cabin_deck'] == '-1').sum()
    df.loc[df['Cabin_deck'] == '-1', 'Cabin_deck'] = rng.choice(
        ['F', 'G'], n_miss, p=[0.5, 0.5])
    df['Cabin_num'] = pd.to_numeric(df['Cabin_num'], errors='coerce')
    mean_num = df['Cabin_num'].mean()
    df['Cabin_num'] = df['Cabin_num'].fillna(int(mean_num)).astype(int)
    n_miss = (df['Cabin_side'] == '-1').sum()
    df.loc[df['Cabin_side'] == '-1', 'Cabin_side'] = rng.choice(
        ['S', 'P'], n_miss, p=[0.5, 0.5])
    df['Cabin_side'] = df['Cabin_side'].map({'S': 0, 'P': 1})

    # Age：在 mean ± 1·std 范围内均匀采样
    mean_age, std_age = df['Age'].mean(), df['Age'].std()
    nan_mask = df['Age'].isna()
    df.loc[nan_mask, 'Age'] = rng.uniform(
        mean_age - std_age, mean_age + std_age, nan_mask.sum())

    # CryoSleep：有消费一定不是冷冻，剩下的填 False
    df.loc[df['CryoSleep'].isna() & (df['TotalSpending'] != 0), 'CryoSleep'] = False
    df['CryoSleep'] = df['CryoSleep'].fillna(False)

    # 冷冻的人消费一律改 0
    spend_cols = base_cols + ['TotalSpending']
    for c in spend_cols:
        df.loc[(df['CryoSleep'] == True) & df[c].isna(), c] = 0.0

    # 如果其他消费列都是 0，缺失的那列也填 0
    for col in base_cols:
        others = [c for c in base_cols if c != col]
        zero_others = (df[others] == 0).all(axis=1)
        df.loc[zero_others & df[col].isna(), col] = 0.0

    # 剩下还缺的，用中位数填
    for c in base_cols:
        df[c] = df[c].fillna(df[c].median())
    df['TotalSpending'] = df[base_cols].sum(axis=1)

    # log 变换（0 替换为 0.367 避免 log(0)）
    for c in spend_cols:
        df.loc[df[c] == 0, c] = 0.367
        df[c] = np.log(df[c])

    df['VIP'] = df['VIP'].fillna(False)
    df['CryoSleep'] = df['CryoSleep'].astype(bool)

    # Name 字段做 label encoding（保留为整数特征，捕捉家庭信号）
    df['Name_first'] = LabelEncoder().fit_transform(df['Name_first'])
    df['Name_last']  = LabelEncoder().fit_transform(df['Name_last'])

    # HomePlanet / Destination / Cabin_deck 做 one-hot
    df = pd.concat([df, pd.get_dummies(df[['HomePlanet','Destination','Cabin_deck']])], axis=1)
    df.drop(columns=['HomePlanet','Destination','Cabin_deck'], inplace=True)

    df_tr_p = df.iloc[:len(df_tr)].copy()
    df_te_p = df.iloc[len(df_tr):].copy()
    df_tr_p['Transported'] = df_tr_p['Transported'].astype(bool)

    y_p   = df_tr_p['Transported']
    X_p   = df_tr_p.drop(columns=['Transported'])
    X_pte = df_te_p.drop(columns=['Transported'])
    return X_p, y_p, X_pte


In [ ]:
# 训练 Voter 2：跑一次 baseline 取 top-27 特征，再用 RandomSearch 最优参拟合
if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('final_spaceship_submission.csv'):
    np.random.seed(42)
    X_p, y_p, X_pte = p5t6_prepare(seed=42)

    # 85/15 切分
    X_tr_p, X_va_p, y_tr_p, y_va_p = train_test_split(
        X_p, y_p, test_size=0.15, random_state=42)

    # 第一遍：用默认 CatBoost 算特征重要性
    imp_model = CatBoostClassifier(verbose=False, allow_writing_files=False)
    imp_model.fit(X_tr_p, y_tr_p)
    imp_df = pd.DataFrame({'feature': X_tr_p.columns,
                           'importance': imp_model.feature_importances_})
    # 取前 27 个最重要的特征
    final_features = list(
        imp_df.sort_values('importance', ascending=False).head(27)['feature'])
    print(f'Voter 2 选出的 top-27 特征（按 CatBoost importance）：')
    for f in final_features:
        print(f'  • {f}')

    final_train = X_tr_p[final_features]
    final_test  = X_pte[final_features]

    # 第二遍：用 RandomSearch 最优参 + top-27 特征，正式训练
    cb_p2 = CatBoostClassifier(**P5T6_PARAMS, verbose=False, allow_writing_files=False)
    cb_p2.fit(final_train, y_tr_p)
    y_pred_p = cb_p2.predict(final_test).astype(bool).ravel()

    pd.DataFrame({'PassengerId': X_pte.index, 'Transported': y_pred_p}
                 ).to_csv('final_spaceship_submission.csv', index=False)
    print(f'\nVoter 2 完成，已保存 final_spaceship_submission.csv （{len(y_pred_p)} 行）')
else:
    print('final_spaceship_submission.csv 已存在，跳过重训。')
    print('如需强制重训，请把第 0 节的 RUN_VOTERS_FROM_SCRATCH 改为 True。')


## 3. Voter 3 — Reference CatBoost（Kaggle 参考实现 + 手调参数）

设计理念：**第三方的稳定基准**。
- FE 和 Voter 1 接近，但更基础，没什么花样
- 调参：完全不调，直接抄 Kaggle top-3 公开 notebook 的默认值（lr=0.01, depth=6, l2=3, iter=2000）
- 训练协议：70/30 单切分 + StandardScaler，单次拟合
- 角色：第三个声音，**防止 Lean 和 Rich 互相回声**

In [ ]:
# Kaggle 参考 notebook 的手调默认值（最稳但最普通的一组参数）
BASELINE_PARAMS = dict(
    iterations=2000,
    learning_rate=0.01,
    depth=6,
    l2_leaf_reg=3,
    loss_function='Logloss',
    eval_metric='Accuracy',
    verbose=False,
)


def baseline_prepare():
    """Voter 3 的基础 FE：跟 Voter 1 类似，但流程更简单"""
    train_raw = pd.read_csv('train.csv')
    test_raw  = pd.read_csv('test.csv')

    test_raw['Transported'] = np.nan
    df = pd.concat([train_raw, test_raw], axis=0)

    # 拆 Cabin
    df[['Deck','Num','Side']] = df['Cabin'].str.split('/', expand=True)
    df['TotalSpend'] = df[SPEND_COLS].sum(axis=1)
    df['Group']      = df['PassengerId'].apply(lambda x: x.split('_')[0])
    df['Group_Size'] = df.groupby('Group')['Group'].transform('count')

    # HomePlanet / Destination：先同组众数填，再全局众数兜底
    for c in ['HomePlanet', 'Destination']:
        grp_mode = df.groupby('Group')[c].transform(
            lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
        df[c] = df[c].fillna(grp_mode)
        df[c] = df[c].fillna(df[c].mode()[0])

    # CryoSleep：有消费 -> False，剩下用众数
    df.loc[df['CryoSleep'].isna() & (df['TotalSpend'] > 0), 'CryoSleep'] = False
    df['CryoSleep'] = df['CryoSleep'].fillna(df['CryoSleep'].mode()[0])

    # Cabin 缺失的整行填占位符
    df['Cabin'] = df['Cabin'].fillna('U/0/U')
    df['Deck'] = df['Cabin'].apply(lambda x: x.split('/')[0])
    df['Side'] = df['Cabin'].apply(lambda x: x.split('/')[2])

    # 消费列 + 派生特征
    df[SPEND_COLS] = df[SPEND_COLS].fillna(0)
    df['TotalSpend']  = df[SPEND_COLS].sum(axis=1)
    df['NoSpending']  = (df['TotalSpend'] == 0).astype(int)
    df['LuxurySpend'] = df['Spa'] + df['VRDeck']
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['VIP'] = df['VIP'].fillna(df['VIP'].mode()[0])
    df['Num'] = pd.to_numeric(df['Num'], errors='coerce')
    df['Num'] = df['Num'].fillna(df['Num'].median())

    df['Group_Size'] = df.groupby('Group')['Group'].transform('count')
    df['Is_Child']   = (df['Age'] <= 12).astype(int)
    # log 版本的消费列
    for c in SPEND_COLS + ['TotalSpend']:
        df[f'{c}_Log'] = np.log1p(df[c])

    test_id_b = test_raw['PassengerId'].copy()
    df.drop(columns=['Cabin','Name','Group','PassengerId'], inplace=True)
    # 类别列 one-hot
    df = pd.get_dummies(df, columns=['HomePlanet','Destination','Side','Deck'])
    for c in ['CryoSleep','VIP']:
        df[c] = df[c].astype(int)

    df_tr_b = df[df['Transported'].notna()].copy()
    df_te_b = df[df['Transported'].isna()].copy().drop(columns=['Transported'])
    df_tr_b['Transported'] = df_tr_b['Transported'].astype(int)
    return df_tr_b, df_te_b, test_id_b


In [ ]:
# 训练 Voter 3：70/30 单切分 + StandardScaler，单次拟合
if RUN_VOTERS_FROM_SCRATCH or not os.path.exists('submission_with_cady_new.csv'):
    df_tr_b, df_te_b, test_id_b = baseline_prepare()
    y_b = df_tr_b['Transported']
    X_b = df_tr_b.drop(columns=['Transported'])

    # 70/30 切分
    X_tr_b, X_va_b, y_tr_b, y_va_b = train_test_split(
        X_b, y_b, test_size=0.3, random_state=42)
    # 标准化
    sc_b = StandardScaler()
    X_tr_b_s = sc_b.fit_transform(X_tr_b)
    df_te_b_s = sc_b.transform(df_te_b)

    # 默认参数训练
    cb_b = CatBoostClassifier(**BASELINE_PARAMS, random_seed=42, allow_writing_files=False)
    cb_b.fit(X_tr_b_s, y_tr_b)
    y_pred_b = cb_b.predict(df_te_b_s).astype(bool).ravel()

    pd.DataFrame({'PassengerId': test_id_b, 'Transported': y_pred_b}
                 ).to_csv('submission_with_cady_new.csv', index=False)
    print(f'Voter 3 完成，已保存 submission_with_cady_new.csv （{len(y_pred_b)} 行）')
else:
    print('submission_with_cady_new.csv 已存在，跳过重训。')
    print('如需强制重训，请把第 0 节的 RUN_VOTERS_FROM_SCRATCH 改为 True。')


## 4. 硬投票合并 — 2-of-3 多数票

读三个 voter 的提交 CSV，对每行做布尔多数票（至少 2 个 voter 预测 True 即为 True），最终输出 `submission_v10_majority.csv`。

这就是 Public LB **0.81529** 的最终提交文件。

最后还会打印每个 voter 和最终结果的一致率 —— 一致率越接近、voter 之间分歧越小（也就越没必要做 ensemble）。我们这里三个 voter 的一致率都在 94-96% 之间，意味着有 4-6% 的"硬样本"靠多数票纠正。

In [ ]:
# 读三个 voter 的 submission
V9_CSV   = 'submission_v9.csv'                     # Voter 1 Lean
SN_CSV   = 'final_spaceship_submission.csv'        # Voter 2 Rich
BASE_CSV = 'submission_with_cady_new.csv'          # Voter 3 Reference

v9_sub   = pd.read_csv(V9_CSV).rename(columns={'Transported': 'v9'})
sn_sub   = pd.read_csv(SN_CSV).rename(columns={'Transported': 'sn'})
base_sub = pd.read_csv(BASE_CSV).rename(columns={'Transported': 'base'})

# 三个 voter 按 PassengerId 对齐
merged = v9_sub.merge(sn_sub, on='PassengerId').merge(base_sub, on='PassengerId')

# 2-of-3 多数票：True 数量 >= 2 即为 True
votes = merged['v9'].astype(int) + merged['sn'].astype(int) + merged['base'].astype(int)
merged['Transported'] = votes >= 2

# 保存最终提交文件
final = merged[['PassengerId', 'Transported']]
final.to_csv('submission_v10_majority.csv', index=False)

# 诊断：每个 voter 和最终结果的一致率（用来量化 voter 之间的差异度）
total = len(merged)
agree_v9   = int((merged['v9']   == merged['Transported']).sum())
agree_sn   = int((merged['sn']   == merged['Transported']).sum())
agree_base = int((merged['base'] == merged['Transported']).sum())

print(f'已保存 submission_v10_majority.csv  (Public LB 0.81529)')
print(f'测试集行数              : {total}')
print(f'Voter 1 (Lean) 与多数票一致  : {agree_v9}  ({agree_v9/total*100:.1f}%)')
print(f'Voter 2 (Rich) 与多数票一致  : {agree_sn}  ({agree_sn/total*100:.1f}%)')
print(f'Voter 3 (Ref)  与多数票一致  : {agree_base}  ({agree_base/total*100:.1f}%)')
final.head()
